# Step 12 · contaminated-registry test + E5 round 2 (closes the remaining future-work items)
1. `registry_contam.py` (DeBERTa, seeds 42–44): does registry-relative calibration hold when the registry already
   contains poisoned tools? Naive contamination (K = 1–3) with max / approved-only / median anchors, plus an adaptive
   "anchor ratchet" attacker and an anchor cap (~25 min).
2. `adv_train.py` (E5, seeds 42–44): **E5 round-2** adversarial training on 15% + 30% white-box rewrites; saves the
   seed-42 checkpoint (~35 min).
3. `attack_whitebox.py --set whitebox.calibrated=true`: adaptive white-box attack on the **calibrated E5 final system**
   (30% budget, 100 steps) + agent check (~40–60 min).
4. `calibrate_adv.py` (E5, seeds 42–44): E5 final-system FPR / detection with error bars (~30 min).
5. `registry_contam.py` (E5, seeds 42–44) (~20 min).

Total ≈ 2.5–3 h. **Settings:** GPU **T4 x2** · Internet **On** · Secrets **Hugging Face** + **mcp-git** ·
Save Version → Save & Run All. Then Output → Download All, put the zip in the repo folder.

In [ ]:
# ---- settings for this run ----
REPO = "github.com/Arn15/mcp-poison-detect"
BRANCH = "main"
E5 = "intfloat/multilingual-e5-base"
SEEDS = "[42,43,44]"
# E5 round 2 = round-1 data (15% rewrites from wb1 + wb1e5) + 30% rewrites found against E5 base and E5 round 1
SRC_TRAIN_E5 = "[results/wb1/whitebox.jsonl,results/wb1e5/whitebox.jsonl,results/wb3_base_b30_e5/whitebox.jsonl,results/wb3_adv_b30_e5/whitebox.jsonl]"
SRC_CAL_E5 = "[results/wb1,results/wb1e5,results/wb3_base_b30_e5,results/wb3_adv_b30_e5]"
ATTACK = "whitebox.max_frac=0.30 whitebox.steps=100 whitebox.calibrated=true"
KEEP_CKPT = False          # True keeps the ~1 GB E5 checkpoint in the output
GPU_DEPS = True
MIN_COMMIT_FILE = "scripts/registry_contam.py"

In [ ]:
import os, subprocess, base64
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
def secret(k):
    try: return sec.get_secret(k)
    except Exception: return None
HF_SECRET, GH_SECRET = "Hugging Face", "mcp-git"   # names of your Kaggle secrets
os.environ["HF_TOKEN"] = secret(HF_SECRET) or ""
gh = secret(GH_SECRET)
assert gh, "Secret 'mcp-git' not attached to this notebook (Add-ons -> Secrets)"
WORK = "/kaggle/working"

# Token goes in an HTTP header for this command only: never in the URL, .git/config, or error output.
b64 = base64.b64encode(f"x-access-token:{gh}".encode()).decode()
auth = ["-c", f"http.extraHeader=Authorization: Basic {b64}"]
def git(*args, cwd=None):
    r = subprocess.run(["git", *auth, *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(gh, "***")
        raise RuntimeError(f"git {args[0]} failed: {msg.strip()}")
    return r.stdout

os.chdir(WORK)
repo_dir = os.path.join(WORK, "mcp-poison-detect")
if os.path.isdir(os.path.join(repo_dir, ".git")):
    git("fetch", "-q", cwd=repo_dir); git("checkout", "-q", BRANCH, cwd=repo_dir); git("pull", "-q", cwd=repo_dir)
else:
    git("clone", "-q", "-b", BRANCH, f"https://{REPO}.git", repo_dir)
os.chdir(repo_dir)
print("commit:", git("log", "-1", "--oneline", cwd=repo_dir))
assert os.path.exists(MIN_COMMIT_FILE), f"{MIN_COMMIT_FILE} missing: push the latest commit first (needs scripts/registry_contam.py)"

In [ ]:
!pip install -q -r requirements.txt
if GPU_DEPS:
    !pip install -q -r requirements-gpu.txt
!pip install -q -e . --no-deps

In [ ]:
import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.device_count() >= 2, "Need GPU T4 x2"
# 1. contaminated-registry test, DeBERTa final system
!python scripts/registry_contam.py --tag contam_deb --set 'contam.seeds={SEEDS}'

In [ ]:
# 2. E5 round-2 adversarial training (3 seeds, global-threshold numbers; seed-42 ADV checkpoint saved)
!python scripts/adv_train.py --tag adv2_e5 --set detector.backbone={E5} 'advtrain.sources={SRC_TRAIN_E5}' 'advtrain.seeds={SEEDS}'

In [ ]:
# 3. adaptive white-box attack on the calibrated E5 final system
import glob, os
adv_run = sorted(glob.glob("/kaggle/working/outputs/adv_train/*adv2_e5"))[-1]
ckpt = os.path.join(adv_run, "ckpt_adv")
assert os.path.exists(os.path.join(ckpt, "detector.json")), ckpt
!python scripts/attack_whitebox.py --tag wb5_e5_final_cal_b30 --set detector.backbone={E5} whitebox.detector_ckpt={ckpt} {ATTACK}

In [ ]:
# 4. E5 final system (round 2 + calibration), 3 seeds
for s in (42, 43, 44):
    !python scripts/calibrate_adv.py --tag cal_adv2_e5_s{s} --set detector.backbone={E5} 'caladv.adv_sources={SRC_CAL_E5}' caladv.seed={s}

In [ ]:
# 5. contaminated-registry test, E5 final system
!python scripts/registry_contam.py --tag contam_e5 --set detector.backbone={E5} 'contam.seeds={SEEDS}'

In [ ]:
# summary
import glob, json, os, shutil
O = "/kaggle/working/outputs"
for r in sorted(glob.glob(f"{O}/registry_contam/*")):
    print(open(f"{r}/contam.md").read())
for r in sorted(glob.glob(f"{O}/adv_train/*adv2_e5")):
    print(open(f"{r}/adv_results.md").read())
run = sorted(glob.glob(f"{O}/attack_whitebox/*wb5_e5_final_cal_b30"))[-1]
s = json.load(open(f"{run}/summary.json"))
print("E5 FINAL-SYSTEM ATTACK:", os.path.basename(run), "| calibrated", s.get("calibrated"), "| evaded", s["n_evaded"],
      "| genuine", s.get("n_genuine_evasions"), "| of", s["n_items"])
for r in sorted(glob.glob(f"{O}/calibrate_adv/*cal_adv2_e5*")):
    print(open(f"{r}/cal_adv.md").read().split("## Target 5%")[0])
if not KEEP_CKPT:
    for d in glob.glob(f"{O}/**/ckpt_*", recursive=True):
        shutil.rmtree(d, ignore_errors=True)
    print("checkpoints removed (KEEP_CKPT=False)")